# Master Results Notebook for Paper Production

This notebook serves as the primary driver for generating the final, publication-quality figures and tables for the paper. It leverages the consolidated logic in `src/sed_pipeline/` to ensure zero data loss and reproducible results.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the project root is in the path to import the local package
sys.path.append(os.path.abspath('..'))

from src.sed_pipeline import config, data_io, composite_math, photometry, visualization, analysis

# Styling for publication
plt.style.use('default')
sns.set_context("paper", font_scale=1.5)
os.makedirs(config.PROCESSED_DATA_DIR, exist_ok=True)

## 1. Load Configurations and Base Models

We start by loading the necessary filters (UVJ, ugr, IRAC) and the base Type 1 and Type 2 AGN SKIRTOR models using parameters defined in the thesis methodology.

In [ ]:
# Load Filters
filters = photometry.load_passbands(config.FILTER_PATHS)

# Set specific paths for templates
skirtor_dir = os.path.join(config.RAW_DATA_DIR, 'Templates', 'Skirtor')
brown_dir = os.path.join(config.RAW_DATA_DIR, 'Templates', 'Brown')

# Load AGN Models
agn_type1 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE1_PARAMS)
agn_type2 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE2_PARAMS)

# Load GALSEDATLAS (Brown) Templates
brown_templates, brown_names = data_io.read_brown_galaxy_templates(brown_dir)

## 2. Generate Composite SEDs and Calculate Colours

We will iterate over the alpha values, calculate composite SEDs for all templates and both AGN types, and compute their colours.

In [ ]:
results = []
alphas = config.ALPHA_VALUES

for t_idx, gal_sed in enumerate(brown_templates):
    gal_name = brown_names[t_idx]
    
    for agn_name, agn_model in [('Type1', agn_type1), ('Type2', agn_type2)]:
        for alpha in alphas:
            comp_sed = composite_math.create_composite_sed(agn_model, gal_sed, alpha)
            
            # Calculate UVJ
            uv, vj = photometry.calculate_UVJ_colours(comp_sed, filters['U'], filters['V'], filters['J'])
            
            # Note: For ugr and IRAC, you'll use astSED to calculate colours similarly.
            # Example (assuming astSED syntax): 
            # ug = astSED.SED.calcColour(sed_obj, filters['u'], filters['g'], magType='AB')
            
            results.append({
                'Template': gal_name,
                'AGN_Type': agn_name,
                'Alpha': alpha,
                'U-V': uv,
                'V-J': vj
                # Add u-g, g-r, I1-I2, I3-I4 here as well
            })

df_results = pd.DataFrame(results)
print("Processed all composite combinations.")
df_results.head()

## 3. Simple Colour-Evolution Tracks

Clean, evolution-only plots (lines tracking $\alpha=0$ to $\alpha=1$) without background scatter clutter.

In [ ]:
def plot_simple_uvj_evolution(df, agn_type, title):
    fig, ax = plt.subplots(figsize=(8, 8))
    
    subset = df[df['AGN_Type'] == agn_type]
    
    for template in subset['Template'].unique():
        temp_df = subset[subset['Template'] == template].sort_values('Alpha')
        ax.plot(temp_df['V-J'], temp_df['U-V'], marker='', linestyle='-', alpha=0.3)
        # Mark start and end
        ax.scatter(temp_df['V-J'].iloc[0], temp_df['U-V'].iloc[0], color='blue', s=20)
        ax.scatter(temp_df['V-J'].iloc[-1], temp_df['U-V'].iloc[-1], color='red', s=20)
        
    visualization.plot_uvj_diagram([], [], ax=ax, title=title)
    
    # Custom legend elements
    from matplotlib.lines import Line2D
    custom_lines = [Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10),
                    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10)]
    ax.legend(custom_lines, ['Pure Galaxy ($\alpha=0$)', 'Max AGN ($\alpha=1$)'], loc='lower right')
    
    plt.tight_layout()
    return fig, ax

fig1, _ = plot_simple_uvj_evolution(df_results, 'Type1', 'Type 1 AGN UVJ Evolution (Tracks Only)')
fig1.savefig(os.path.join(config.PROCESSED_DATA_DIR, "Paper_UVJ_Evolution_Type1_Tracks.pdf"), bbox_inches='tight')

fig2, _ = plot_simple_uvj_evolution(df_results, 'Type2', 'Type 2 AGN UVJ Evolution (Tracks Only)')
fig2.savefig(os.path.join(config.PROCESSED_DATA_DIR, "Paper_UVJ_Evolution_Type2_Tracks.pdf"), bbox_inches='tight')
plt.show()

## 4. Full Visualizations (Figures)
Plotting the full Type 1 and Type 2 AGN UVJ Colour Evolutions with density/scatter (e.g., `Figure \ref{fig:Brown-Type1Composite_UVJ}`).

In [ ]:
# Example of a full density plot for a specific alpha, or all overlaid
# ... (Add full density plot code using visualization.plot_uvj_diagram(..., show_density=True))

## 5. Statistical Tables (LaTeX Exports)
Calculating the mean vector offsets for UVJ across alphas, and exporting to LaTeX.

In [ ]:
def calculate_offsets(df, agn_type):
    subset = df[df['AGN_Type'] == agn_type]
    offsets = []
    for alpha in alphas:
        alpha_df = subset[subset['Alpha'] == alpha]
        initial_df = subset[subset['Alpha'] == 0.0]
        
        vj_alpha = alpha_df['V-J'].values
        uv_alpha = alpha_df['U-V'].values
        vj_init = initial_df['V-J'].values
        uv_init = initial_df['U-V'].values
        
        mean_offset = analysis.calculate_mean_vector_offset(vj_alpha, uv_alpha, vj_init, uv_init)
        offsets.append({'Alpha': alpha, 'Mean_Offset': mean_offset})
    return pd.DataFrame(offsets)

offset_type1 = calculate_offsets(df_results, 'Type1')
offset_type2 = calculate_offsets(df_results, 'Type2')

# Combine for the final table
table_df = pd.DataFrame({
    'AGN Contribution ($\alpha$)': [f"{int(a*100)}\%" for a in alphas],
    'Type 1 Mean Offset (dex)': offset_type1['Mean_Offset'].round(3),
    'Type 2 Mean Offset (dex)': offset_type2['Mean_Offset'].round(3)
})

display(table_df)

# Export to CSV
table_df.to_csv(os.path.join(config.PROCESSED_DATA_DIR, 'UVJ_vectoroffset.csv'), index=False)

# Export to LaTeX
latex_table = table_df.to_latex(index=False, caption="Mean vector offset for Type 1 and Type 2 AGN composites in the UVJ diagram.", label="tab:UVJ_vectoroffset")
with open(os.path.join(config.PROCESSED_DATA_DIR, 'UVJ_vectoroffset.tex'), 'w') as f:
    f.write(latex_table)

print("Exported UVJ Vector Offset table to CSV and LaTeX.")